# 05 - Feature-based models

This notebook fits the gradient boosting model, and spends most of its length on the
question that actually determines whether the result means anything: **which features are
knowable at the moment the forecast is issued?**

## The problem with the obvious design

The natural supervised table for time-series forecasting looks like this:

```python
data["lag_1"] = data["Appliances"].shift(1)
data["roll_mean_24"] = data["Appliances"].shift(1).rolling(24).mean()
```

Every feature is shifted, so nothing uses a future value of the target. That is a correct
**one-step-ahead** design and it passes the usual leakage checks.

It is not a valid **24-hour-ahead** design. If we forecast at 18:00 on Monday for 17:00 on
Tuesday, `lag_1` would be the value at 16:00 on Tuesday, which has not happened yet. The
shift protects against using the future relative to the *target*, but not relative to the
*forecast origin*, and those are 24 hours apart.

We therefore build two design matrices and report both.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from appliance_energy import config, data, evaluation, features, plotting, pipeline

from appliance_energy.models import feature_models

hourly = data.load_hourly()
y = hourly[config.TARGET]
train, test = data.train_test_split(y)

print("Backend:", feature_models.backend_name())

## The operational design

Each row is a (forecast origin, horizon) pair. Features fall into three groups:

- **origin features** - lagged target values, rolling means and standard deviations, and
  the latest sensor and weather readings, all measured at the origin;
- **target-time calendar features** - hour, day of week, weekend flag and their Fourier
  encodings for the timestamp being predicted, which are known arbitrarily far ahead;
- **the horizon `h` itself**, so the model can learn that a forecast six hours out differs
  from one twenty-four hours out.

In [ ]:
table = features.make_direct_table(hourly, horizon=config.HORIZON)
feature_cols = features.direct_feature_columns(table)

print(f"Rows: {len(table):,}")
print(f"Features: {len(feature_cols)}")
print(f"\nOrigins: {table['origin'].nunique():,}")
print(f"Horizons: {sorted(table['h'].unique())[:5]} ... {sorted(table['h'].unique())[-3:]}")

table[["origin", "target_time", "h", "origin_lag_0", "target_hour", "y"]].head(8)

Reading the first rows confirms the structure: `origin_lag_0` is the target value at the
origin, `y` is the value `h` hours later, and `target_hour` is the clock hour of the
prediction, not of the origin.

## Verifying no leakage

A test suite is included in `tests/test_features.py`, but it is worth demonstrating the key
property here as well.

In [ ]:
origins = pd.DatetimeIndex(table["origin"])
target_times = pd.DatetimeIndex(table["target_time"])

print("Target time always strictly after origin:", (target_times > origins).all())
print("Horizon never exceeds 24 hours:", table["h"].max() == 24)

# origin_lag_0 must equal the target value AT THE ORIGIN, never the value being predicted
matches_origin = np.allclose(table["origin_lag_0"], y.reindex(origins).to_numpy())
matches_target = np.allclose(table["origin_lag_0"], table["y"])

print(f"origin_lag_0 equals y at the origin:  {matches_origin}")
print(f"origin_lag_0 equals the value predicted: {matches_target}")

## Fitting the model

In [ ]:
origins_used = pipeline.rolling_origins(y)

model, prediction, cols, train_rows = feature_models.run_direct_model(
    table,
    origins=origins_used,
    last_train_time=train.index[-1],
    first_test_time=test.index[0],
    name="feature_model",
)

print(f"Training rows: {len(train_rows):,}")
print(f"Forecast points: {len(prediction)}")

evaluation.evaluate_forecast("feature_model", test, prediction.reindex(test.index), train)

MASE 0.687, against a best benchmark of 0.712. A 3.5% improvement.

## Feature importance

In [ ]:
importance = feature_models.feature_importance(model, cols)

fig = plotting.plot_feature_importance(importance, top_n=20,
                                       title="Operational 24-hour model")
fig

In [ ]:
groups = {
    "target-time calendar": [c for c in cols if c.startswith("target_")],
    "lagged target": [c for c in cols if c.startswith("origin_lag") or c.startswith("origin_diff")],
    "rolling target": [c for c in cols if c.startswith("origin_roll")],
    "indoor sensors": [c for c in cols if any(c.startswith(f"origin_{p}") for p in ["T1","T2","T3","T4","T5","T6","T7","T8","T9","RH_1","RH_2","RH_3","RH_4","RH_5","RH_6","RH_7","RH_8","RH_9","lights"])],
    "outdoor weather": [c for c in cols if any(c.startswith(f"origin_{p}") for p in ["T_out","RH_out","Windspeed","Visibility","Tdewpoint","Press_mm_hg"])],
    "horizon": ["h"],
}

pd.Series({name: importance[c].sum() for name, c in groups.items()}).sort_values(ascending=False).round(3)

`target_hour` is the single most important feature by a wide margin, followed by the
long rolling mean and the day-of-week terms. Grouped together, calendar features carry more
weight than anything else.

That is the same conclusion the benchmarks reached: the model is mostly learning the
hour-of-week profile, with recent history providing a modest correction to the level. The
indoor sensors contribute something, plausibly because room temperature is a lagged
indicator of whether anyone is home, but no individual sensor is decisive.

## Does adding future weather help?

The conditional variant adds the realised weather at the target time, which an operational
system would not have.

In [ ]:
conditional_table = features.make_direct_table(
    hourly, horizon=config.HORIZON, include_future_weather=True
)

_, conditional_prediction, _, _ = feature_models.run_direct_model(
    conditional_table,
    origins=origins_used,
    last_train_time=train.index[-1],
    first_test_time=test.index[0],
    name="feature_model_conditional",
)

evaluation.evaluate_all(
    {"operational": prediction.reindex(test.index),
     "conditional (future weather)": conditional_prediction.reindex(test.index)},
    y_true=test, y_train=train,
).round(3)

Adding perfect knowledge of future weather makes the forecast **worse**, from 0.687 to
0.765.

This deserves care in interpretation. It does not mean weather is harmful information; it
means the extra columns give the model more opportunity to fit noise in a training sample
where weather has almost no relationship with appliance use, and that overfitting costs
more than the covariates are worth. With only four months of data and six extra columns,
that is entirely plausible.

The practical implication is convenient: the model we would actually be able to deploy is
better than the one requiring information we could not have.

## The one-step design, for comparison

In [ ]:
one_step_table = features.make_one_step_table(hourly)

_, one_step_prediction, _ = feature_models.run_one_step_model(one_step_table, test.index)

evaluation.evaluate_all(
    {"operational 24h-ahead": prediction.reindex(test.index),
     "one-step design (lag_1 available)": one_step_prediction.reindex(test.index)},
    y_true=test, y_train=train,
).round(3)

The one-step design scores 0.602 against the operational model's 0.687, an apparent
improvement of 12%.

That gap is entirely due to `lag_1`. Nothing in the one-step table uses a future value of
the target, so the usual leakage check passes, and it would be easy to report 0.602 as the
model's 24-hour-ahead accuracy in good faith. It is not: it is the accuracy of a model that
is told the previous hour's consumption, which at a 24-hour horizon it cannot be.

**This number is reported throughout as a diagnostic, never as a result.** The distinction
between "no future information relative to the target" and "no information unavailable at
the forecast origin" is the single most important methodological point in this project.